# EPİAŞ MCP (PTF) Raw Data Fetch Notebook

The purpose of this notebook is to:

- Obtain a TGT (Ticket Granting Ticket) via EPİAŞ CAS  
- Fetch PTF data from the MCP endpoint  
- Convert the data into a DataFrame  
- Save the data as partitioned parquet files  
- Test the incremental fetch logic  

In [35]:
from __future__ import annotations

import os 
import time
from datetime import datetime,timedelta
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

In [52]:
load_dotenv()

CAS_URL ="https://giris.epias.com.tr/cas/v1/tickets"
MCP_URL ="https://seffaflik.epias.com.tr/electricity-service/v1/markets/dam/data/mcp"

OUT_DIR = Path("../data/notebook_test")
DEFAULT_START_DATE = "2018-01-01"
CHUNK_DAYS = 30
OVERLAP_DAYS = 1
TZ_SUFFIX = "+03:00"

In [37]:
username = os.getenv("EPIAS_USERNAME")
password = os.getenv("EPIAS_PASSWORD")

print("USERNAME OK:",bool(username))
print("PASSWORD OK:",bool(password))

if not username or not password:
    raise RuntimeError("EPIAS_USERNAME or EPIAS_PASSWORD failed")

USERNAME OK: True
PASSWORD OK: True


In [38]:
def get_tgt(username: str, password: str) -> str:
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "text/plain",
    }
    payload = {
        "username": username,
        "password": password,
    }

    r = requests.post(CAS_URL, headers=headers, data=payload, timeout=30)
    r.raise_for_status()

    tgt = r.text.strip()

    if not tgt.startswith("TGT-"):
        raise RuntimeError(
            "Geçerli TGT alınamadı. "
            f"status={r.status_code}, "
            f"content_type={r.headers.get('Content-Type')}, "
            f"text={r.text[:300]}"
        )

    return tgt

In [39]:
tgt = get_tgt(username,password)
print("TGT OK")
print(tgt[:20]+"...")

TGT OK
TGT-8011408-hSkI9pKY...


In [40]:
def fetch_mcp_raw(start_date: str, end_date: str, tgt: str) -> dict:
    headers = {
        "TGT": tgt,
        "Content-Type": "application/json",
    }
    payload = {
        "startDate": f"{start_date}T00:00:00{TZ_SUFFIX}",
        "endDate": f"{end_date}T23:59:59{TZ_SUFFIX}",
    }

    r = requests.post(MCP_URL, headers=headers, json=payload, timeout=60)

    if r.status_code != 200:
        print("MCP status:", r.status_code)
        print("MCP text head:", r.text[:500])

    r.raise_for_status()
    return r.json()

In [41]:
test_start = "2024-01-01"
test_end = "2024-01-03"

raw = fetch_mcp_raw(test_start, test_end, tgt)
type(raw), list(raw.keys())[:10] if isinstance(raw, dict) else "not dict"

(dict, ['items', 'page', 'statistic'])

In [42]:
raw

{'items': [{'date': '2024-01-01T00:00:00+03:00',
   'hour': '00:00',
   'price': 1299.98,
   'priceUsd': 44.16,
   'priceEur': 39.91},
  {'date': '2024-01-01T01:00:00+03:00',
   'hour': '01:00',
   'price': 1299.98,
   'priceUsd': 44.16,
   'priceEur': 39.91},
  {'date': '2024-01-01T02:00:00+03:00',
   'hour': '02:00',
   'price': 1248.54,
   'priceUsd': 42.41,
   'priceEur': 38.33},
  {'date': '2024-01-01T03:00:00+03:00',
   'hour': '03:00',
   'price': 1299.98,
   'priceUsd': 44.16,
   'priceEur': 39.91},
  {'date': '2024-01-01T04:00:00+03:00',
   'hour': '04:00',
   'price': 1200,
   'priceUsd': 40.76,
   'priceEur': 36.84},
  {'date': '2024-01-01T05:00:00+03:00',
   'hour': '05:00',
   'price': 1200,
   'priceUsd': 40.76,
   'priceEur': 36.84},
  {'date': '2024-01-01T06:00:00+03:00',
   'hour': '06:00',
   'price': 1248.54,
   'priceUsd': 42.41,
   'priceEur': 38.33},
  {'date': '2024-01-01T07:00:00+03:00',
   'hour': '07:00',
   'price': 1299.99,
   'priceUsd': 44.16,
   'priceEur

In [43]:
def find_first_list(obj):
    if isinstance(obj,list):
        return obj

    if isinstance(obj,dict):
        for v in obj.values():
            found = find_first_list(v)
            if found is not None:
                return found
    return None

In [44]:
def raw_to_df(raw: dict, chunk_start: str, chunk_end: str) -> pd.DataFrame:
    rows = find_first_list(raw)

    if not isinstance(rows, list) or len(rows) == 0:
        return pd.DataFrame()

    df = pd.json_normalize(rows)

    df["_chunk_start"] = chunk_start
    df["_chunk_end"] = chunk_end

    for col in ["date", "datetime", "time", "period", "periodDate", "periodTime"]:
        if col in df.columns:
            try:
                df[col] = pd.to_datetime(df[col], errors="coerce")
            except Exception:
                pass

    return df

In [45]:
df_test = raw_to_df(raw,test_start,test_end)
print(df_test.shape)
print(df_test.columns.tolist())
df_test.head()

(72, 7)
['date', 'hour', 'price', 'priceUsd', 'priceEur', '_chunk_start', '_chunk_end']


,date,hour,price,priceUsd,priceEur,_chunk_start,_chunk_end
0,2024-01-01 00:00:00+03:00,00:00,1299.98,44.16,39.91,2024-01-01,2024-01-03
1,2024-01-01 01:00:00+03:00,01:00,1299.98,44.16,39.91,2024-01-01,2024-01-03
2,2024-01-01 02:00:00+03:00,02:00,1248.54,42.41,38.33,2024-01-01,2024-01-03
3,2024-01-01 03:00:00+03:00,03:00,1299.98,44.16,39.91,2024-01-01,2024-01-03
4,2024-01-01 04:00:00+03:00,04:00,1200.00,40.76,36.84,2024-01-01,2024-01-03


In [46]:
def detect_timestamp_column(df: pd.DataFrame) -> str | None:
    for c in ["date", "datetime", "time", "period", "periodDate", "periodTime"]:
        if c in df.columns and pd.api.types.is_datetime64_any_dtype(df[c]):
            return c
    return None

In [47]:
ts_col = detect_timestamp_column(df_test)
print("Detected timestamp column:", ts_col)

Detected timestamp column: date


In [53]:
def write_parquet_partitioned(df:pd.DataFrame,out_dir:str |Path)-> int:
    if df.empty:
        return o
    df = df.copy()
    out_dir = Path(out_dir)

    ts_col = detect_timestamp_column(df)

    if ts_col is not None:
        df["year"] =df[ts_col].dt.year.astype("int16")
        df["month"] = df[ts_col].dt.month.astype("int8")
    else:
        cs = pd.to_datetime(df["_chunk_start"].iloc[0],errors="coerce")
        if pd.isna(cs):
            raise ValueError("Year/month could not be extracted for the partition.")
        df["year"] = int(cs.year)
        df["month"] = int(cs.month)

    out_dir.mkdir(parents=True,exist_ok=True)

    df.to_parquet(
        out_dir,
        engine="pyarrow",
        compression ="snappy",
        index =False,
        partition_cols=["year","month"]
    )

    return len(df)



In [54]:
written=write_parquet_partitioned(df_test,OUT_DIR)
print("Written rows",written)
print("Output dir exists:",OUT_DIR.exists())

Written rows 72
Output dir exists: True


In [56]:
df_saved = pd.read_parquet(OUT_DIR)
print(df_saved.shape)
df_saved.head()

(72, 9)


,date,hour,price,priceUsd,priceEur,_chunk_start,_chunk_end,year,month
0,2024-01-01 00:00:00+03:00,00:00,1299.98,44.16,39.91,2024-01-01,2024-01-03,2024,1
1,2024-01-01 01:00:00+03:00,01:00,1299.98,44.16,39.91,2024-01-01,2024-01-03,2024,1
2,2024-01-01 02:00:00+03:00,02:00,1248.54,42.41,38.33,2024-01-01,2024-01-03,2024,1
3,2024-01-01 03:00:00+03:00,03:00,1299.98,44.16,39.91,2024-01-01,2024-01-03,2024,1
4,2024-01-01 04:00:00+03:00,04:00,1200.00,40.76,36.84,2024-01-01,2024-01-03,2024,1


In [58]:
def get_last_timestamp_from_parquet_dataset(path: str | Path) -> pd.Timestamp | None:
    path = Path(path)

    if not path.exists():
        return None

    try:
        df = pd.read_parquet(path)

        if df.empty:
            return None

        ts_col = None
        for c in ["date", "datetime", "time", "period", "periodDate", "periodTime"]:
            if c in df.columns:
                ts_col = c
                break

        if ts_col is None:
            return None

        df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
        df = df.dropna(subset=[ts_col])

        if df.empty:
            return None

        return df[ts_col].max()

    except Exception as e:
        print(f"WARNING: Existing parquet dataset failed: {e}")
        return None

In [59]:
def resolve_fetch_window(
    out_dir:str | Path,
    default_start_date:str,
    overlap_days : int =1
) -> tuple[str,str]:
    last_ts = get_last_timestamp_from_parquet_dataset(out_dir)

    if last_ts is None:
        start_date = default_start_date
        print("No existing raw dataset. Full backfill will begin.")
    
    else:
        overlap_start = last_ts - pd.Timedelta(days=overlap_days)
        start_date = overlap_start.strftime("%Y-%m-%d")
        print(f"Last timestamp found: {last_ts}")
        print(f"Overlap days: {overlap_days}")

    end_date = pd.Timestamp.now().strftime("%Y-%m-%d")
    return start_date , end_date


In [60]:
start_date, end_date = resolve_fetch_window(
    out_dir=OUT_DIR,
    default_start_date=DEFAULT_START_DATE,
    overlap_days=OVERLAP_DAYS,
)

print("Resolved window:", start_date, "->", end_date)

Last timestamp found: 2024-01-03 23:00:00+03:00
Overlap days: 1
Resolved window: 2024-01-02 -> 2026-04-08


In [61]:
def daterange_chunks(start_date : str , end_date:str ,chunk_days:int):
    start = datetime.strptime(start_date,"%Y-%m-%d").date()
    end = datetime.strptime(end_date,"%Y-%m-%d").date()

    cur = start

    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days -1),end)
        yield cur.strftime("%Y-%m-%d"),chunk_end.strftime("%Y-%m-%d")
        cur = chunk_end + timedelta(days=1)

In [67]:
chunks = list(daterange_chunks("2024-01-01", "2024-03-15", chunk_days=30))
chunks[:30], len(chunks)

([('2024-01-01', '2024-01-30'),
  ('2024-01-31', '2024-02-29'),
  ('2024-03-01', '2024-03-15')],
 3)

In [69]:
demo_start = "2024-01-01"
demo_end = "2024-02-15"

ok_chunks = 0
fail_chunks = 0
total_rows = 0

for s,e in daterange_chunks(demo_start,demo_end,chunk_days=CHUNK_DAYS):
    try:
        print(f"Fetching:{s}->{e}")

        raw = fetch_mcp_raw(s,e,tgt)
        df = raw_to_df(raw,s,e)

        if df.empty:
            print("No rows returned.")
            ok_chunks +=1
            continue

        written = write_parquet_partitioned(df,OUT_DIR)
        print(f"  WROTE {written} rows")

        ok_chunks +=1

        time.sleep(0.2)

    except requests.HTTPError as ex:
        fail_chunks += 1
        print(f"  FAILED HTTPError: {ex}")

    except Exception as ex:
        fail_chunks += 1
        print(f"  FAILED Error: {ex}")

print("DONE")
print("OK_CHUNKS =", ok_chunks)
print("FAIL_CHUNKS =", fail_chunks)
print("TOTAL_ROWS =", total_rows)

Fetching:2024-01-01->2024-01-30
  WROTE 720 rows
Fetching:2024-01-31->2024-02-15
  WROTE 384 rows
DONE
OK_CHUNKS = 2
FAIL_CHUNKS = 0
TOTAL_ROWS = 0


In [70]:
df_all = pd.read_parquet(OUT_DIR)

print(df_all.shape)
print(df_all.columns.tolist())

ts_col = detect_timestamp_column(df_all)
print("timestamp col:", ts_col)

if ts_col:
    print("min timestamp:", df_all[ts_col].min())
    print("max timestamp:", df_all[ts_col].max())

df_all.head()

(1104, 9)
['date', 'hour', 'price', 'priceUsd', 'priceEur', '_chunk_start', '_chunk_end', 'year', 'month']
timestamp col: date
min timestamp: 2024-01-01 00:00:00+03:00
max timestamp: 2024-02-15 23:00:00+03:00


,date,hour,price,priceUsd,priceEur,_chunk_start,_chunk_end,year,month
0,2024-01-01 00:00:00+03:00,00:00,1299.98,44.16,39.91,2024-01-01,2024-01-30,2024,1
1,2024-01-01 01:00:00+03:00,01:00,1299.98,44.16,39.91,2024-01-01,2024-01-30,2024,1
2,2024-01-01 02:00:00+03:00,02:00,1248.54,42.41,38.33,2024-01-01,2024-01-30,2024,1
3,2024-01-01 03:00:00+03:00,03:00,1299.98,44.16,39.91,2024-01-01,2024-01-30,2024,1
4,2024-01-01 04:00:00+03:00,04:00,1200.00,40.76,36.84,2024-01-01,2024-01-30,2024,1


In [71]:
if ts_col:
    temp = df_all.copy()
    temp[ts_col] = pd.to_datetime(temp[ts_col], errors="coerce")
    temp = temp.dropna(subset=[ts_col]).sort_values(ts_col)

    print("duplicate timestamp count:", temp[ts_col].duplicated().sum())
    print("null rows:", temp[ts_col].isna().sum())

duplicate timestamp count: 0
null rows: 0
